<a href="https://colab.research.google.com/github/shiv1ant/Food-Delivery-Operation-Analytics/blob/main/Project8_Food_Delivery_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 8 — Food Delivery Operations Analytics
**Industry:** FoodTech / Logistics  |  **Tools:** Python, pandas, NumPy (no plotting libraries — tables and text only)

## Business scenario
A food-delivery platform wants to evaluate **delivery times, order outcomes, restaurant performance and city-level operations** in order to identify concrete service improvements.

## Objective
Work through assignment tasks **109 – 123**: load and clean the order data, measure delivery performance and order outcomes, rank restaurants and cities, detect unusually long deliveries, and finish with five data-backed operational recommendations.

## How to run on Google Colab
1. Unzip the project on your computer, then upload **`08_food_delivery.csv`** and this notebook to Colab (or upload the whole `.zip` — the loader cell unzips it automatically).
2. Click **Runtime → Run all**.
3. If the CSV is not found, the loader cell opens an upload dialog.

> Pandas and NumPy are pre-installed on Colab, so nothing needs to be installed.

## Notebook map
| Section | Task(s) |
|---|---|
| Setup & loading | 109 |
| Data quality | 110 (missing delivery times), 111 (duplicates) |
| Order outcomes | 112 |
| City & restaurant performance | 113, 114, 115 |
| Demand patterns | 116 |
| Delivery-time analysis | 117, 118, 119, 120, 121 |
| Revenue | 122 |
| Recommendations | 123 |
| Extra analysis | Bonus section (beyond required tasks) |

## Dataset description
The file `08_food_delivery.csv` is a **synthetic (fictional) dataset** created for this assignment (generator script included: `generate_dataset.py`, fixed random seed). It deliberately contains realistic data-quality problems — missing delivery times, duplicate rows, inconsistent city labels and extreme deliveries — so that every task has something meaningful to find.

| Column | Type | Description |
|---|---|---|
| `Order_ID` | text | Order identifier (should be unique) |
| `Order_DateTime` | datetime | When the order was placed |
| `City` | text | City where the order was delivered |
| `Restaurant` | text | Restaurant that prepared the order |
| `Cuisine` | text | Cuisine category of the restaurant |
| `Order_Value` | number (₹) | Order amount in Indian rupees |
| `Payment_Mode` | text | UPI, Credit Card, Debit Card, Cash on Delivery or Wallet |
| `Distance_km` | number | Restaurant-to-customer distance |
| `Delivery_Time_min` | number | Minutes from order placed to delivered (blank if never delivered/logged) |
| `Order_Status` | text | `Delivered`, `Delayed` or `Cancelled` |
| `Customer_Rating` | number | 1–5 rating (blank for cancelled orders) |

## Definitions & assumptions used throughout
* **Delayed order:** an order whose status is `Delayed` (delivery took more than 45 minutes).
* **Cancellation rate / delay rate:** number of cancelled (or delayed) orders ÷ total orders in that city × 100.
* **Revenue:** `Order_Value` of orders that were **not cancelled** (a cancelled order brings no revenue).
* **Delivery-time analysis** uses only orders that actually reached the customer.
* **Delivery bands:** Fast ≤ 30 min, Standard 31–45 min, Delayed > 45 min.

In [ ]:
# Setup: only pandas and NumPy are used. No plotting libraries are imported.
import os, glob, zipfile
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# Locate the dataset (works on Colab and locally).
CSV_NAME = "08_food_delivery.csv"

def find_csv():
    for pattern in [CSV_NAME, f"/content/{CSV_NAME}", f"/content/**/{CSV_NAME}", f"**/{CSV_NAME}"]:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    return None

path = find_csv()

# If only a .zip was uploaded, extract it and look again.
if path is None:
    for z in glob.glob("*.zip") + glob.glob("/content/*.zip"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(".")
        path = find_csv()
        if path:
            break

# Last resort: ask the user to upload the file (Colab only).
if path is None:
    try:
        from google.colab import files
        files.upload()
        path = find_csv()
    except ImportError:
        pass

assert path is not None, f"Could not find {CSV_NAME}. Upload it next to this notebook."
print("Using file:", path)

---
# Task 109 — Load and inspect the order dataset
**Goal:** understand the structure, data types and obvious problems before any analysis.

**Approach:** read the CSV, look at shape / columns / types / summary statistics, and check the categorical columns for inconsistent labels.

In [ ]:
raw = pd.read_csv(path)
df = raw.copy()                       # work on a copy; `raw` stays untouched for reference

print("Shape (rows, columns):", df.shape)
print("\nColumn data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
print("Summary of numeric columns:")
display(df.describe().T)

print("\nUnique values per column:")
display(df.nunique().to_frame("Unique_Values"))

print("\nRaw city labels (note the inconsistent spelling/case):")
display(df["City"].value_counts().to_frame("Rows"))

**Cleaning step 1 – standardise text and types.** The raw city labels contain trailing spaces and different capitalisation (e.g. `"Delhi "`, `"MUMBAI"`) which would split one city into several groups. We trim whitespace, apply title-case and convert `Order_DateTime` to a real datetime.

In [ ]:
for col in ["City", "Restaurant", "Cuisine", "Payment_Mode", "Order_Status"]:
    df[col] = df[col].str.strip()
df["City"] = df["City"].str.title()
df["Order_DateTime"] = pd.to_datetime(df["Order_DateTime"])

print("City labels after cleaning:")
display(df["City"].value_counts().to_frame("Rows"))
print("Order period:", df["Order_DateTime"].min(), "→", df["Order_DateTime"].max())

---
# Task 110 — Find missing delivery times and decide how to handle them
**Goal:** quantify missing values and choose a defensible treatment.

**Approach:**
1. Count missing values per column.
2. Look at the **status** of orders with a missing delivery time — *why* is it missing?
3. Apply a treatment that depends on the reason.

In [ ]:
missing = df.isna().sum().to_frame("Missing")
missing["Missing_%"] = (missing["Missing"] / len(df) * 100).round(2)
display(missing)

mask_missing = df["Delivery_Time_min"].isna()
print(f"Orders with missing delivery time: {mask_missing.sum()} ({mask_missing.mean():.1%} of all rows)\n")
print("Status of those orders:")
display(df.loc[mask_missing, "Order_Status"].value_counts().to_frame("Orders"))

### Decision
Missing delivery times come from **two different causes**, so they get **two different treatments**:

| Cause | Treatment | Reason |
|---|---|---|
| **Cancelled orders** | **Leave blank** | The food never reached the customer, so there is no delivery time. Filling a number would be inventing data. |
| **Delivered / Delayed orders** (logging failure) | **Impute with the median** delivery time of the same *City + Order_Status* group | The median is robust to extreme values, and using the status keeps imputed values consistent (a `Delayed` order must not be given a 30-minute "fast" time). Falls back to the city median if a group is empty. |

The original column is preserved; the treated values go into a new column `Delivery_Time_Clean`, and `Delivery_Imputed` flags every filled row for transparency.

In [ ]:
def add_clean_delivery(frame):
    """Create Delivery_Time_Clean + Delivery_Imputed using the rule described above."""
    out = frame.copy()
    is_missing = out["Delivery_Time_min"].isna()
    not_cancelled = out["Order_Status"] != "Cancelled"
    to_fill = is_missing & not_cancelled

    med_city_status = out.groupby(["City", "Order_Status"])["Delivery_Time_min"].transform("median")
    med_city = out.groupby("City")["Delivery_Time_min"].transform("median")

    out["Delivery_Time_Clean"] = out["Delivery_Time_min"]
    out.loc[to_fill, "Delivery_Time_Clean"] = med_city_status[to_fill].fillna(med_city[to_fill])
    out["Delivery_Imputed"] = to_fill
    return out

df = add_clean_delivery(df)

print("Imputed (logging failures):         ", int(df["Delivery_Imputed"].sum()))
print("Left blank (cancelled, no delivery):", int((df["Delivery_Time_Clean"].isna()).sum()))
print("\nExample of imputed rows:")
display(df.loc[df["Delivery_Imputed"], ["Order_ID", "City", "Order_Status", "Delivery_Time_min", "Delivery_Time_Clean"]].head())

---
# Task 111 — Check duplicate `Order_ID` values
**Goal:** every order should appear exactly once. Find and resolve repeats.

**Approach:** distinguish between
* **exact duplicates** – every column identical (system double-logging) → safe to drop;
* **conflicting duplicates** – same `Order_ID` but different content (e.g. a corrected record) → must be reviewed, not silently deleted.

In [ ]:
dup_id_mask = df["Order_ID"].duplicated(keep=False)
exact_dup_mask = df.duplicated(keep="first")

print("Rows whose Order_ID appears more than once:", int(dup_id_mask.sum()))
print("Distinct Order_IDs that are repeated:      ", df.loc[dup_id_mask, "Order_ID"].nunique())
print("Exact duplicate rows (identical in every column):", int(exact_dup_mask.sum()))

# Step 1: remove exact duplicates
df_nodup = df.drop_duplicates(keep="first")

# Step 2: what is left? Same Order_ID but different values = conflicting records
conflicts = df_nodup[df_nodup["Order_ID"].duplicated(keep=False)].sort_values("Order_ID")
print("\nConflicting records (same Order_ID, different content):", conflicts["Order_ID"].nunique(), "IDs")
display(conflicts[["Order_ID", "City", "Restaurant", "Order_Value", "Order_Status"]])

### Decision
* **Exact duplicates are dropped** (45 double-logged rows).
* **Conflicting records:** the dataset has no "last updated" timestamp, so we cannot tell which version is correct. We **keep the first occurrence** and list the conflicts above for manual verification against the source system. This affects only a handful of orders, so the impact on the results is negligible.
* The imputation column is rebuilt afterwards so that medians are not influenced by the duplicates.

In [ ]:
df = df_nodup.drop_duplicates(subset="Order_ID", keep="first").reset_index(drop=True)
df = add_clean_delivery(df.drop(columns=["Delivery_Time_Clean", "Delivery_Imputed"]))

print("Rows before cleaning:", len(raw))
print("Rows after cleaning: ", len(df))
print("Is Order_ID unique now?", df["Order_ID"].is_unique)

---
# Task 112 — Total orders and order-status percentages
**Approach:** count unique `Order_ID`s, then use `value_counts(normalize=True)` for the status mix.

In [ ]:
total_orders = df["Order_ID"].nunique()
print("Total orders:", f"{total_orders:,}")

status_counts = df["Order_Status"].value_counts()
status_pct = df["Order_Status"].value_counts(normalize=True) * 100
status_summary = pd.DataFrame({"Orders": status_counts, "Percent": status_pct.round(2)})
display(status_summary)

In [ ]:
print(f"Insight: {status_pct['Delivered']:.1f}% of orders were delivered on time, "
      f"{status_pct['Delayed']:.1f}% were delayed and {status_pct['Cancelled']:.1f}% were cancelled. "
      f"Together, {status_pct['Delayed'] + status_pct['Cancelled']:.1f}% of orders had a service problem.")

---
# Task 113 — Average delivery time by city
**Approach:** `groupby("City")` on `Delivery_Time_Clean` (cancelled orders are automatically excluded because they have no delivery time).

In [ ]:
city_delivery = (df.groupby("City")["Delivery_Time_Clean"]
                   .agg(Orders_Delivered="count", Avg_Delivery_Min="mean", Median_Delivery_Min="median")
                   .sort_values("Avg_Delivery_Min", ascending=False))
display(city_delivery)

slow, fast = city_delivery.index[0], city_delivery.index[-1]
gap = city_delivery.loc[slow, "Avg_Delivery_Min"] - city_delivery.loc[fast, "Avg_Delivery_Min"]
print(f"Insight: {slow} is the slowest city ({city_delivery.loc[slow, 'Avg_Delivery_Min']:.1f} min) and {fast} the fastest "
      f"({city_delivery.loc[fast, 'Avg_Delivery_Min']:.1f} min) — a gap of {gap:.1f} minutes per order.")

---
# Task 114 — Five restaurants with the highest total order value
**Approach:** `groupby("Restaurant")["Order_Value"].sum()`, sorted, top 5. The first table follows the task literally (all orders placed). The second shows the same ranking using only **fulfilled** orders, i.e. real revenue.

In [ ]:
top5_all = (df.groupby("Restaurant")["Order_Value"].agg(Orders="count", Total_Order_Value="sum")
              .sort_values("Total_Order_Value", ascending=False).head(5))
print("Top 5 restaurants by total order value (all orders placed):")
display(top5_all)

top5_fulfilled = (df[df["Order_Status"] != "Cancelled"].groupby("Restaurant")["Order_Value"]
                    .agg(Fulfilled_Orders="count", Fulfilled_Value="sum")
                    .sort_values("Fulfilled_Value", ascending=False).head(5))
print("\nSame ranking using fulfilled (non-cancelled) orders only:")
display(top5_fulfilled)

---
# Task 115 — Average order value by restaurant
**Approach:** `groupby("Restaurant")["Order_Value"].mean()` together with order counts, so averages can be read in context.

In [ ]:
rest_aov = (df.groupby("Restaurant")["Order_Value"]
              .agg(Orders="count", Avg_Order_Value="mean", Total_Order_Value="sum")
              .sort_values("Avg_Order_Value", ascending=False))
display(rest_aov)

print(f"Insight: highest average order value is {rest_aov.index[0]} (₹{rest_aov['Avg_Order_Value'].iloc[0]:,.0f}); "
      f"lowest is {rest_aov.index[-1]} (₹{rest_aov['Avg_Order_Value'].iloc[-1]:,.0f}). "
      f"Overall average across all orders: ₹{df['Order_Value'].mean():,.0f}.")

---
# Task 116 — Identify peak ordering hours
**Approach:** extract the hour from `Order_DateTime`, count orders per hour, and flag **peak hours** as those with order volume above **1.5 × the average hourly volume**.

In [ ]:
df["Order_Hour"] = df["Order_DateTime"].dt.hour

hourly = df.groupby("Order_Hour")["Order_ID"].count().to_frame("Orders")
hourly["Share_%"] = (hourly["Orders"] / hourly["Orders"].sum() * 100).round(2)
threshold = 1.5 * hourly["Orders"].mean()
hourly["Peak_Hour"] = hourly["Orders"] > threshold

print(f"Average orders per hour: {hourly['Orders'].mean():.0f} | peak threshold (1.5×): {threshold:.0f}\n")
display(hourly)

In [ ]:
peak_hours = hourly[hourly["Peak_Hour"]].index.tolist()
print("Top 5 busiest hours:")
display(hourly.sort_values("Orders", ascending=False).head(5)[["Orders", "Share_%"]])

peak_share = hourly.loc[peak_hours, "Share_%"].sum()
print(f"Insight: peak hours are {peak_hours} (24-hour clock). These {len(peak_hours)} hours alone produce {peak_share:.1f}% of all orders.")

---
# Task 117 — Compare delivery time across order-status categories
**Approach:** group by `Order_Status` and summarise `Delivery_Time_Clean`. Cancelled orders have no delivery time, so they show a count of 0.

In [ ]:
status_delivery = (df.groupby("Order_Status")["Delivery_Time_Clean"]
                     .agg(Orders_With_Time="count", Mean="mean", Median="median", Std="std", Min="min", Max="max"))
display(status_delivery)

d, l = status_delivery.loc["Delivered"], status_delivery.loc["Delayed"]
print(f"Insight: delivered orders average {d['Mean']:.1f} min vs {l['Mean']:.1f} min for delayed orders "
      f"({l['Mean'] - d['Mean']:.1f} min longer). Cancelled orders have no delivery time because they never reached the customer.")

---
# Task 118 — Cancellation and delay rates by city
**Definitions:** rate = (cancelled or delayed orders in the city) ÷ (all orders in the city) × 100.
**Approach:** `pivot_table` of status counts per city, then convert to percentages.

In [ ]:
status_by_city = pd.pivot_table(df, index="City", columns="Order_Status",
                                values="Order_ID", aggfunc="count", fill_value=0)
status_by_city["Total_Orders"] = status_by_city.sum(axis=1)
status_by_city["Cancellation_Rate_%"] = status_by_city["Cancelled"] / status_by_city["Total_Orders"] * 100
status_by_city["Delay_Rate_%"] = status_by_city["Delayed"] / status_by_city["Total_Orders"] * 100
status_by_city["Problem_Rate_%"] = status_by_city["Cancellation_Rate_%"] + status_by_city["Delay_Rate_%"]

city_rates = status_by_city.sort_values("Problem_Rate_%", ascending=False)
display(city_rates[["Total_Orders", "Cancelled", "Delayed", "Cancellation_Rate_%", "Delay_Rate_%", "Problem_Rate_%"]])

c_max = city_rates["Cancellation_Rate_%"].idxmax()
d_max = city_rates["Delay_Rate_%"].idxmax()
print(f"Insight: highest cancellation rate → {c_max} ({city_rates.loc[c_max, 'Cancellation_Rate_%']:.1f}%); "
      f"highest delay rate → {d_max} ({city_rates.loc[d_max, 'Delay_Rate_%']:.1f}%).")

---
# Task 119 — Median and standard deviation of delivery time (NumPy)
**Approach:** convert the delivery times of dispatched orders into a NumPy array and use `np.median` and `np.std`. We report both the **population** (`ddof=0`) and **sample** (`ddof=1`) standard deviation; for a dataset this large they are practically identical.

In [ ]:
times = df["Delivery_Time_Clean"].dropna().to_numpy()      # NumPy array, cancelled orders excluded

np_mean   = np.mean(times)
np_median = np.median(times)
np_std_p  = np.std(times, ddof=0)
np_std_s  = np.std(times, ddof=1)

print(f"Number of delivery times : {times.size:,}")
print(f"Mean                     : {np_mean:.2f} min")
print(f"Median (np.median)       : {np_median:.2f} min")
print(f"Std dev, population      : {np_std_p:.2f} min")
print(f"Std dev, sample (ddof=1) : {np_std_s:.2f} min")

# cross-check with pandas
print("\nCross-check with pandas -> median:", df["Delivery_Time_Clean"].median(), "| std:", round(df["Delivery_Time_Clean"].std(), 2))
print(f"\nInsight: the median ({np_median:.0f} min) is slightly below the mean ({np_mean:.1f} min) — a sign that a few very long deliveries pull the average up.")

---
# Task 120 — Identify unusually long deliveries using the IQR method
**Method:**
* `Q1` = 25th percentile, `Q3` = 75th percentile, `IQR = Q3 − Q1`
* Upper fence = `Q3 + 1.5 × IQR` → any delivery above this is an **outlier (unusually long)**.
* Lower fence = `Q1 − 1.5 × IQR` (checked for completeness).

In [ ]:
q1, q3 = np.percentile(times, [25, 75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1 = {q1:.1f} | Q3 = {q3:.1f} | IQR = {iqr:.1f}")
print(f"Lower fence = {lower_fence:.1f} min | Upper fence = {upper_fence:.1f} min")

delivered = df[df["Delivery_Time_Clean"].notna()]
long_deliveries = delivered[delivered["Delivery_Time_Clean"] > upper_fence]
too_fast = delivered[delivered["Delivery_Time_Clean"] < lower_fence]

print(f"\nUnusually long deliveries: {len(long_deliveries)} ({len(long_deliveries) / len(delivered):.2%} of delivered orders)")
print(f"Unusually short deliveries: {len(too_fast)}")
print("\nThe 10 longest deliveries:")
display(long_deliveries.sort_values("Delivery_Time_Clean", ascending=False)
        [["Order_ID", "City", "Restaurant", "Distance_km", "Order_Hour", "Delivery_Time_Clean", "Order_Status"]].head(10))

In [ ]:
outlier_city = (long_deliveries.groupby("City")["Order_ID"].count().to_frame("Outliers")
                .join(delivered.groupby("City")["Order_ID"].count().rename("Delivered_Orders")))
outlier_city["Outlier_Rate_%"] = outlier_city["Outliers"] / outlier_city["Delivered_Orders"] * 100
display(outlier_city.sort_values("Outlier_Rate_%", ascending=False))

print(f"Insight: any delivery slower than {upper_fence:.0f} minutes is statistically unusual. "
      f"{len(long_deliveries)} such orders should be investigated individually (rider issue, wrong address, weather, kitchen delay).")

---
# Task 121 — Create delivery-time bands (Fast / Standard / Delayed)
**Bands:**

| Band | Delivery time |
|---|---|
| Fast | ≤ 30 minutes |
| Standard | 31 – 45 minutes |
| Delayed | > 45 minutes |
| Not Delivered | cancelled orders (no time recorded) |

**Approach:** `np.select()` for the main column (it lets us add the *Not Delivered* class) and `pd.cut()` as an independent cross-check.

In [ ]:
t = df["Delivery_Time_Clean"]
conditions = [t.isna(), t <= 30, t <= 45, t > 45]
labels     = ["Not Delivered", "Fast", "Standard", "Delayed"]
df["Delivery_Band"] = np.select(conditions, labels, default="Unknown")

# Cross-check with pd.cut on dispatched orders
cut_band = pd.cut(t.dropna(), bins=[0, 30, 45, np.inf], labels=["Fast", "Standard", "Delayed"])
agree = (cut_band.astype(str) == df.loc[t.notna(), "Delivery_Band"]).all()
print("np.select and pd.cut give identical bands:", agree)

band_summary = df["Delivery_Band"].value_counts().reindex(labels).to_frame("Orders")
band_summary["Percent"] = (band_summary["Orders"] / band_summary["Orders"].sum() * 100).round(2)
display(band_summary)

print("\nBand mix by city (% of delivered orders):")
band_city = pd.crosstab(df.loc[t.notna(), "City"], df.loc[t.notna(), "Delivery_Band"], normalize="index") * 100
display(band_city[["Fast", "Standard", "Delayed"]].round(1).sort_values("Delayed", ascending=False))

---
# Task 122 — Group revenue by city and payment mode
**Approach:** `pivot_table` with `City` as rows and `Payment_Mode` as columns, summing `Order_Value` of non-cancelled orders, with row/column totals (`margins=True`).

In [ ]:
revenue_df = df[df["Order_Status"] != "Cancelled"]

rev_pivot = pd.pivot_table(revenue_df, index="City", columns="Payment_Mode", values="Order_Value",
                           aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
rev_pivot = rev_pivot.sort_values("Total", ascending=False)
print("Revenue (₹) by city and payment mode:")
display(rev_pivot)

print("\nPayment-mode share of revenue within each city (%):")
share = rev_pivot.drop(index="Total").drop(columns="Total")
share = share.div(share.sum(axis=1), axis=0) * 100
display(share.round(1))

In [ ]:
total_rev = rev_pivot.loc["Total", "Total"]
top_city = rev_pivot.drop(index="Total")["Total"].idxmax()
pay_totals = rev_pivot.loc["Total"].drop("Total").sort_values(ascending=False)
print(f"Insight: total revenue is ₹{total_rev:,.0f}. {top_city} contributes the most "
      f"(₹{rev_pivot.loc[top_city, 'Total']:,.0f}, {rev_pivot.loc[top_city, 'Total'] / total_rev:.1%}). "
      f"{pay_totals.index[0]} is the leading payment mode ({pay_totals.iloc[0] / total_rev:.1%} of revenue).")

---
# Bonus analysis (beyond the required tasks)
Extra tables that help explain *why* performance differs and feed the recommendations in Task 123.

In [ ]:
# B1. Delivery time: peak vs off-peak hours
df["Period"] = np.where(df["Order_Hour"].isin(peak_hours), "Peak hours", "Off-peak hours")
period_summary = (df.groupby("Period")
                    .agg(Orders=("Order_ID", "count"),
                         Avg_Delivery_Min=("Delivery_Time_Clean", "mean"),
                         Delay_Rate_pct=("Order_Status", lambda s: (s == "Delayed").mean() * 100),
                         Cancel_Rate_pct=("Order_Status", lambda s: (s == "Cancelled").mean() * 100)))
print("B1. Peak vs off-peak performance:")
display(period_summary)

# B2. Does distance explain delivery time?
dd = df.dropna(subset=["Delivery_Time_Clean"])
slope, intercept = np.polyfit(dd["Distance_km"], dd["Delivery_Time_Clean"], 1)
corr = np.corrcoef(dd["Distance_km"], dd["Delivery_Time_Clean"])[0, 1]
print(f"\nB2. Each extra km adds about {slope:.2f} minutes (correlation = {corr:.2f}).")

In [ ]:
# B3. Restaurant reliability (restaurants with at least 150 orders)
rest_quality = (df.groupby("Restaurant")
                  .agg(Orders=("Order_ID", "count"),
                       Avg_Delivery_Min=("Delivery_Time_Clean", "mean"),
                       Delay_Rate_pct=("Order_Status", lambda s: (s == "Delayed").mean() * 100),
                       Cancel_Rate_pct=("Order_Status", lambda s: (s == "Cancelled").mean() * 100),
                       Avg_Rating=("Customer_Rating", "mean")))
rest_quality = rest_quality[rest_quality["Orders"] >= 150].sort_values("Delay_Rate_pct", ascending=False)
print("B3. Restaurants ranked by delay rate:")
display(rest_quality.round(2))

# B4. Customer rating by order outcome
print("\nB4. Average customer rating by order status:")
display(df.groupby("Order_Status")["Customer_Rating"].agg(Rated_Orders="count", Avg_Rating="mean"))

In [ ]:
# B5. City scorecard: rank 1 = best on each metric, then average the ranks
scorecard = city_delivery[["Avg_Delivery_Min"]].join(city_rates[["Cancellation_Rate_%", "Delay_Rate_%"]])
scorecard["Revenue"] = rev_pivot.drop(index="Total")["Total"]
ranks = scorecard[["Avg_Delivery_Min", "Cancellation_Rate_%", "Delay_Rate_%"]].rank(ascending=True)
scorecard["Avg_Rank (1=best)"] = ranks.mean(axis=1)
print("B5. City scorecard (lower Avg_Rank = better operations):")
display(scorecard.sort_values("Avg_Rank (1=best)"))

---
# Task 123 — Five operational recommendations
Each recommendation below is **generated from the results above** (the figures are computed, not typed in), so it always matches the data. Every recommendation follows the pattern: *finding → action → expected benefit*.

In [ ]:
# Pull the facts needed
slowest_city   = city_delivery["Avg_Delivery_Min"].idxmax()
slowest_min    = city_delivery.loc[slowest_city, "Avg_Delivery_Min"]
fastest_min    = city_delivery["Avg_Delivery_Min"].min()
network_median = np_median

peak_txt   = ", ".join(f"{h}:00" for h in peak_hours)
pk, off    = period_summary.loc["Peak hours"], period_summary.loc["Off-peak hours"]

worst_cancel_city = city_rates["Cancellation_Rate_%"].idxmax()
worst_cancel_rate = city_rates["Cancellation_Rate_%"].max()
avg_cancel_rate   = status_pct["Cancelled"]

worst_rest = rest_quality.index[0]
worst_rest_delay = rest_quality["Delay_Rate_pct"].iloc[0]
avg_delay_rate = status_pct["Delayed"]

cod_share = share["Cash on Delivery"].mean() if "Cash on Delivery" in share.columns else float("nan")
cod_city  = share["Cash on Delivery"].idxmax()
upi_share_total = pay_totals.get("UPI", 0) / total_rev * 100

recs = [
 ("1. Add rider capacity in the slowest cities",
  f"{slowest_city} averages {slowest_min:.1f} min per delivery versus {fastest_min:.1f} min in the fastest city, "
  f"and the network median is {network_median:.0f} min. Rebalance rider supply and delivery zones in the slowest cities, "
  f"starting with {slowest_city}, aiming to close at least half of that gap."),
 ("2. Schedule riders and kitchens around peak hours",
  f"Peak ordering hours ({peak_txt}) bring {peak_share:.0f}% of all orders. During peak hours the delay rate is {pk['Delay_Rate_pct']:.1f}% "
  f"vs {off['Delay_Rate_pct']:.1f}% off-peak and cancellations are {pk['Cancel_Rate_pct']:.1f}% vs {off['Cancel_Rate_pct']:.1f}%. "
  f"Use shift incentives, pre-positioned riders and a cap on new-order intake per restaurant at those hours."),
 (f"3. Reduce cancellations in {worst_cancel_city}",
  f"{worst_cancel_city} has the highest cancellation rate ({worst_cancel_rate:.1f}%) against a network average of {avg_cancel_rate:.1f}%. "
  f"Investigate cancellation reasons, show more realistic ETAs at checkout and auto-reassign orders when a restaurant or rider is slow to accept."),
 (f"4. Put {worst_rest} on a restaurant improvement plan",
  f"{worst_rest} has the highest delay rate among high-volume restaurants ({worst_rest_delay:.1f}% vs {avg_delay_rate:.1f}% overall). "
  f"Review its preparation times, introduce a preparation-time SLA and tie visibility in the app to on-time performance."),
 ("5. Investigate extreme deliveries and steer customers to prepaid payments",
  f"{len(long_deliveries)} deliveries exceeded the IQR fence of {upper_fence:.0f} min — review each (address errors, rider breakdowns, weather) "
  f"and set automatic alerts at {upper_fence:.0f} min. Separately, Cash on Delivery averages {cod_share:.0f}% of city revenue "
  f"(highest in {cod_city}) while UPI already brings {upi_share_total:.0f}% of revenue; prepaid-payment offers reduce handling time and cancellation risk."),
]

for title, text in recs:
    print(title)
    print("   " + text + "\n")

---
# Conclusion
This notebook cleaned the order data (standardised labels, treated missing delivery times by cause, removed duplicate records), then measured order outcomes, city and restaurant performance, demand peaks, delivery-time distribution and revenue by payment mode. Unusually long deliveries were identified with the IQR rule, and five recommendations were derived directly from the computed results.

**Limitations:** the data is synthetic; delivery bands and the 45-minute delay threshold are assumptions that a real business would set from service-level agreements; conflicting duplicate IDs were resolved by keeping the first record.